# The Flex Alert scorecard: replication

Energy Research Warehouse (ERW), session 60. This notebook reproduces every number on `/grid/caiso/alerts` from the warehouse's own inputs: the alert days and their hours, the 24 hour-of-day models, every alert day's actual, predicted and reduction figures, the pooled and yearly results, the out-of-sample check on the hottest non-alert days, the five variants' errors and estimates, and the wholesale value. The method is `docs/methods/flex_alert_scorecard.md`; the point estimates here are a second implementation of it, written from the method in numpy and pandas, and asserted equal to the tables `flex_alert_effects` and `flex_alert_model`. The 90 percent intervals come from a bootstrap with a published seed: the last part re-runs it through `warehouse/derived/flex_alert_scorecard.py` and asserts the same intervals (set `ERW_NOTEBOOK_QUICK=1` to skip that part, about five minutes).

Run it from the repository root on a data machine: it reads `warehouse/output/` and the CISO extract under `warehouse/raw/eia930_emissions/`. No network, no model.

In [ ]:
import glob, os
import numpy as np
import pandas as pd

ROOT = os.path.abspath(".") if os.path.isdir("warehouse") else os.path.abspath("..")
OUT = os.path.join(ROOT, "warehouse", "output")

def table(name):
    path = os.path.join(OUT, name + ".csv")
    with open(path, encoding="utf-8") as f:
        n = sum(1 for ln in f if ln.startswith("#"))
    return pd.read_csv(path, skiprows=n)

eff, mod = table("flex_alert_effects"), table("flex_alert_model")
TZ, WEIGHTS, MONTHS = "America/Los_Angeles", {"SAC": 0.30, "FAT": 0.20, "LAX": 0.50}, [5, 6, 7, 8, 9, 10]
DAY = eff[eff["freq"] == "P1D"].pivot(index="ts_utc", columns="variable", values="value")
DAY.index = DAY.index.str[:10]
G = mod[mod["ts_utc"] == "2018-07-01T00:00:00Z"].set_index("variable")["value"]
print(len(DAY), "alert days in the table;", int(G["pooled_hours"]), "alert hours")

## Demand and weather

Demand: EIA-930's hourly Demand for CAISO's balancing authority, from the newest extract of EIA's CISO workbook (each value the hour's mean MW; the extract's `ts_utc` is the hour's start). Hours the starter export's screen marks implausible (below 30 percent of the median hour, or one hour more than 20 percent above or below both neighbours) are set missing.

Weather: `noaa_isd_hourly` at Sacramento, Fresno and Los Angeles; each observation to its nearest UTC hour, the station's hourly mean, gaps of at most two hours interpolated, weighted 0.30, 0.20 and 0.50, and an hour kept only with all three stations.

In [ ]:
ex = sorted(glob.glob(os.path.join(ROOT, "warehouse", "raw", "eia930_emissions", "*", "ciso_hours.csv")))[-1]
with open(ex, encoding="utf-8") as f:
    skip = sum(1 for ln in f if ln.startswith("#"))
x = pd.read_csv(ex, skiprows=skip, usecols=["ts_utc", "demand_mwh"])
load = pd.Series(x["demand_mwh"].to_numpy(float), index=pd.to_datetime(x["ts_utc"], utc=True)).sort_index()
load = load[~load.index.duplicated()]
load = load.reindex(pd.date_range(load.index.min(), load.index.max(), freq="h"))
prev, nxt = load.shift(1), load.shift(-1)
bad = ((load < 0.3 * load.median()) | ((load > 1.2 * prev) & (load > 1.2 * nxt)) | ((load < prev / 1.2) & (load < nxt / 1.2))) & load.notna()
load[bad] = np.nan

w = table("noaa_isd_hourly")
w = w[w["entity"].isin(["noaa:SAC", "noaa:FAT", "noaa:LAX"])].copy()
w["t"] = pd.to_datetime(w["ts_utc"], utc=True).dt.round("h")
wx = {}
for var in ("temperature_f", "dew_point_f"):
    p = w[w["variable"] == var].groupby(["t", "entity"])["value"].mean().unstack("entity")
    p = p.reindex(pd.date_range(p.index.min(), p.index.max(), freq="h")).interpolate(limit=2, limit_area="inside")
    wx[var] = sum(p["noaa:" + c] * k for c, k in WEIGHTS.items())
wx = pd.DataFrame(wx)
wx["cdh"] = (wx["temperature_f"] - 65).clip(lower=0)
wx["cdh24"] = wx["cdh"].shift(1).rolling(24, min_periods=20).mean()
print(int(bad.sum()), "demand hours screened out;", int(wx["temperature_f"].notna().sum()), "weighted weather hours")

## The panel

Every UTC hour of May to October, 2018-07-01 to 2025-10-31, with its local (Pacific) day and hour, day type (weekday, Saturday, Sunday), US federal holiday, month and year. A day enters the fit only with every local hour present and every value held. The alert days are those of the table (their hours too); a day with a CAISO notice of an alert type is never a training day.

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

idx = load.index.intersection(wx.index)
P = pd.DataFrame({"y": load.reindex(idx), "temp": wx["temperature_f"].reindex(idx), "dew": wx["dew_point_f"].reindex(idx),
                  "cdh": wx["cdh"].reindex(idx), "cdh24": wx["cdh24"].reindex(idx)})
loc = idx.tz_convert(TZ)
P["ts"], P["day"], P["hour"], P["month"], P["year"], P["dow"] = idx, loc.strftime("%Y-%m-%d"), loc.hour, loc.month, loc.year, loc.dayofweek
P = P[(P["day"] >= "2018-07-01") & (P["day"] <= "2025-10-31") & P["month"].isin(MONTHS)].reset_index(drop=True)
hol = set(USFederalHolidayCalendar().holidays("2018-01-01", "2025-12-31").strftime("%Y-%m-%d"))
P["holiday"] = P["day"].isin(hol).astype(float)
P["sat"], P["sun"] = (P["dow"] == 5).astype(float), (P["dow"] == 6).astype(float)
ok = P.dropna(subset=["y", "cdh", "cdh24", "dew"]).groupby("day").size()
need = P.groupby("day").size()
full = set(ok.index[(ok == need.reindex(ok.index)) & (ok >= 23)])

notices = table("caiso_grid_emergencies")
alert_types = {"flex_alert", "eea_watch", "eea1", "eea2", "eea3", "alert", "warning", "stage1", "stage2", "stage3", "load_interruption"}
alerts = list(DAY.index)
assert set(alerts) <= full
hours = {d: list(range(int(DAY.loc[d, "alert_first_hour"]), int(DAY.loc[d, "alert_last_hour"]))) for d in alerts}
assert all(len(hours[d]) == int(DAY.loc[d, "alert_hours"]) for d in alerts)
# every alert day of the table is a day with an ISO-wide notice of an alert type (or within a multi-day notice's span)
nt = notices[notices["event_type"].isin(alert_types) & notices["x_region"].fillna("").isin(["ISO", "CAISO Grid", ""])]
# a notice with parsed hours covers every day from its start to its end; one without covers its own day, except the row
# dated 2020-09-30 whose text, "10/1/2020 15:00:00 through 22:00:00", names 2020-10-01 (the method's text forms)
with_span = nt.dropna(subset=["x_start", "x_end"])
span_days = {d.strftime("%Y-%m-%d") for s, e in zip(with_span["x_start"], with_span["x_end"]) for d in pd.date_range(s[:10], e[:10])}
own_days = set(nt[nt["x_start"].isna()]["event_date"].str[:10]) - {"2020-09-30"} | {"2020-10-01"}
excluded = span_days | own_days
assert set(alerts) <= excluded
train = sorted(full - excluded - set(alerts))
print(len(alerts), "alert days,", len(train), "training days")
assert len(train) == int(G["train_days"])

## The model: one least-squares fit per local hour

For hour h: demand = a + b1 CDH + b2 CDH^2 + b3 CDH24 + b4 CDH24^2 + b5 dew + b6 CDH x dew + Saturday + Sunday + holiday + month (May the base) + year (2018 the base). CDH is the cooling degree hour at 65 F of the weighted temperature, CDH24 the mean CDH over the 24 hours before, dew the weighted dew point. The variants leave out terms (`VARIANTS`).

In [ ]:
TERMS = ["cdh", "cdh2", "cdh24", "cdh24_2", "dew", "cdh_dew"]
VARIANTS = {"minimal": ["cdh", "cdh2"], "base": ["cdh", "cdh2", "cdh24", "dew"], "base_prev24sq": ["cdh", "cdh2", "cdh24", "cdh24_2", "dew"],
            "base_cdh_dew": ["cdh", "cdh2", "cdh24", "dew", "cdh_dew"], "chosen": TERMS}
P["cdh2"], P["cdh24_2"], P["cdh_dew"] = P["cdh"] ** 2, P["cdh24"] ** 2, P["cdh"] * P["dew"]

def X(q, terms):
    cols = [np.ones(len(q))] + [q[t].to_numpy(float) for t in terms] + [q[c].to_numpy(float) for c in ("sat", "sun", "holiday")]
    cols += [(q["month"] == m).to_numpy(float) for m in MONTHS[1:]] + [(q["year"] == y).to_numpy(float) for y in range(2019, 2026)]
    return np.column_stack(cols)

def fit(days, terms):
    q = P[P["day"].isin(days)]
    return {h: np.linalg.lstsq(X(g, terms), g["y"].to_numpy(float), rcond=None)[0] for h, g in q.groupby("hour")}

def pred(q, b, terms):
    out = np.full(len(q), np.nan)
    M = X(q, terms)
    for h, coef in b.items():
        i = (q["hour"] == h).to_numpy()
        out[i] = M[i] @ coef
    return out

B = fit(train, TERMS)
A = P[P["day"].isin(alerts)].copy()
A["pred"] = pred(A, B, TERMS)
A["alert"] = [h in hours[d] for d, h in zip(A["day"], A["hour"])]
a = A[A["alert"]]
mine = a.assign(red=a["pred"] - a["y"]).groupby("day").agg(actual_mwh=("y", "sum"), predicted_mwh=("pred", "sum"), reduction_mwh=("red", "sum"), reduction_mw=("red", "mean"))
for col in mine.columns:
    assert np.allclose(mine[col], DAY.loc[mine.index, col], atol=0.01), col
print("every alert day's actual, predicted and reduction equal the table's")
mine.sort_values("reduction_mw", ascending=False).round(1)

## Pooled and by year

The pooled reduction is the mean over every alert hour of every alert day (MW), and its sum (MWh); a year's, the same over its days.

In [ ]:
red = a["pred"] - a["y"]
assert abs(red.mean() - G["pooled_reduction_mw"]) < 0.01 and abs(red.sum() - G["pooled_reduction_mwh"]) < 0.1
assert abs(red.sum() / a["pred"].sum() * 100 - G["pooled_reduction_pct"]) < 1e-3
Y = eff[eff["freq"] == "P1Y"].pivot(index="ts_utc", columns="variable", values="value")
for y, g in a.assign(red=red).groupby(a["day"].str[:4]):
    assert abs(g["red"].mean() - Y.loc[f"{y}-01-01T00:00:00Z", "year_reduction_mw"]) < 0.01, y
print(f"pooled: {red.mean():.1f} MW over {len(a)} alert hours; every year's estimate equals the table's")

## The check: the hottest non-alert days, out of sample

The hottest tenth of training days (by the day's highest weighted temperature) is split into five folds at random (seed 20261002); each fold is predicted by the model fitted on every other training day, and the error (actual less predicted) in the hours 16:00 to 21:00 summarised. The same, for every variant.

In [ ]:
tmax = P[P["day"].isin(train)].groupby("day")["temp"].max()
k = int(round(len(tmax) * 0.10))
hot = sorted(tmax.sort_values(ascending=False).index[:k])
assert abs(tmax[hot].min() - G["heldout_threshold_temp_f"]) < 1e-3

def cv(terms):
    h = np.array(hot)
    np.random.default_rng(20261002).shuffle(h)
    parts = []
    for f in np.array_split(h, 5):
        b = fit(sorted(set(train) - set(f)), terms)
        q = P[P["day"].isin(f)].copy()
        q["err"] = q["y"] - pred(q, b, terms)
        parts.append(q)
    q = pd.concat(parts)
    return q[q["hour"].isin([16, 17, 18, 19, 20])]

w = cv(TERMS)
assert abs(w["err"].mean() - G["oos_bias_mw"]) < 0.01
assert abs(np.sqrt((w["err"] ** 2).mean()) - G["oos_rmse_mw"]) < 0.01
assert abs((w["err"].abs() / w["y"]).mean() * 100 - G["oos_mape_pct"]) < 1e-3
for name, terms in VARIANTS.items():
    wv = cv(terms)
    assert abs(wv["err"].mean() - G[f"variant_{name}_oos_bias_mw"]) < 0.01, name
    bv = fit(train, terms)
    av = A.copy(); av["p"] = pred(av, bv, terms); av = av[av["alert"]]
    assert abs((av["p"] - av["y"]).mean() - G[f"variant_{name}_pooled_reduction_mw"]) < 0.01, name
print(f"out of sample on {len(hot)} hot days: bias {w['err'].mean():.1f} MW, RMSE {np.sqrt((w['err'] ** 2).mean()):.1f} MW; every variant equals the table")

## The wholesale value

Each alert hour's reduction (MWh, the hour's mean MW for one hour) times that hour's day-ahead price, the mean of SP15 and NP15 (an hour counts only with both hubs), summed over the hours that have a price.

In [ ]:
pr = table("caiso_dam_alert_day_hub_prices")
t = pd.to_datetime(pr["ts_utc"], utc=True).dt.tz_convert(TZ)
pr = pr.assign(day=t.dt.strftime("%Y-%m-%d"), hour=t.dt.hour).groupby(["day", "hour"]).agg(v=("value", "mean"), n=("entity", "nunique"))
pr = pr[pr["n"] == 2]["v"]
v = a.assign(red=red, price=[pr.get((d, h), np.nan) for d, h in zip(a["day"], a["hour"])]).dropna(subset=["price"])
assert len(v) == int(G["pooled_value_hours"])
assert abs((v["red"] * v["price"]).sum() - G["pooled_value_usd"]) < 1.0
print(f"value over {len(v)} priced alert hours: USD {(v['red'] * v['price']).sum():,.0f}")

## The intervals

The 90 percent intervals combine a block bootstrap over calendar weeks of the training days (all 24 hours refitted, 1,000 replicates) with one out-of-fold residual day per alert day (centred by hour), drawn with seed 20261002. Re-running the published estimator with that seed gives the table's intervals exactly.

In [ ]:
if os.environ.get("ERW_NOTEBOOK_QUICK") != "1":
    import sys
    sys.path.insert(0, os.path.join(ROOT, "warehouse", "derived"))
    import flex_alert_scorecard as fa
    inp = fa.inputs(lambda *_: None)
    _, _, vec = fa.out_of_sample(inp["p"], inp["eligible"], inp["hold"])
    prices, _ = fa.read_prices()
    D, H, Yr, pooled, _ = fa.estimate(inp["p"], inp["alerts"], inp["eligible"], vec, prices)
    for c in ("reduction_mw_lo", "reduction_mw_hi", "reduction_mwh_lo", "reduction_mwh_hi"):
        assert np.allclose(D.set_index("day")[c], DAY.loc[D["day"], c], atol=0.01), c
        assert abs(pooled[c] - G["pooled_" + c]) < 0.01, c
    print("every interval equals the table's")